In [1]:
import sys
sys.path.append('../')
import transport_frames.src.graph_builder.criteria as criteria
import transport_frames.src.metrics.indicators as indicators
import transport_frames.src.metrics.grade_territory as grade_territory
import transport_frames.src.graph_builder.graphbuilder as graphbuilder
import momepy
import osmnx as ox
import geopandas as gpd
import shapely
from shapely.geometry import Point
import shapely
from shapely import wkt
import pandas as pd
import networkx as nx
import numpy as np
import json
from dongraphio import DonGraphio, GraphType
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def read_json_file(filename):
    with open(filename, 'r') as f:
        json_object = json.load(f)

    result_dict = {}

    for key in json_object.keys():
        if isinstance(json_object[key], str):  # check if the value is a string
            try:
                gdf = gpd.read_file(json_object[key])
                result_dict[key] = gdf
            except:
                try:
                    result_dict[key] = json.loads(json_object[key])
                except json.JSONDecodeError:
                    result_dict[key] = json_object[key]
        else:
            result_dict[key] = json_object[key]

    return result_dict

a = read_json_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/indicators/Тюменская_область/Тюменская_область_indicators_gpsp.json')

print(a.keys()) # check all possible keys


dict_keys(['aggregated_density', 'road_length_gdf', 'connectivity', 'connectivity_public_transport', 'to_fed_roads', 'to_region_admin_center', 'azs_availability', 'train_stops_availability', 'international_aero_availability', 'local_aero_availability', 'port_availability', 'number_of_bus_routes', 'number_of_bus_stops', 'number_of_fuel_stations', 'number_of_train_stops', 'number_of_international_aero', 'number_of_local_aero', 'number_of_ports', 'train_paths_length'])


In [1]:
a['to_fed_roads']

NameError: name 'a' is not defined

In [275]:
aggregated_density = a['aggregated_density'].rename(columns={'density': 'road_density'}) # density
road_length_gdf = a['road_length_gdf'] # reg1_length reg2_length reg3_length	
to_fed_roads = a['to_fed_roads'].rename(columns={'density': 'road_density'}) # to_service
to_region_admin_center = a['to_region_admin_center'].rename(columns={'to_service': 'to_region_admin_center'}) # to_service
connectivity = a['connectivity'].rename(columns={'to_service': 'connectivity'})
# connectivity_public_transport = a['connectivity_public_transport'].rename(columns={'to_service': 'connectivity_public_transport'})
azs_availability = a['azs_availability'].rename(columns={'to_service': 'azs_availability'}) # to_service
number_of_fuel_stations = a['number_of_fuel_stations'].rename(columns={'service_count': 'number_of_fuel_stations'}) # service_count
train_stops_availability = a['train_stops_availability'].rename(columns={'to_service': 'train_stops_availability'}) # to_service
number_of_train_stops = a['number_of_train_stops'].rename(columns={'service_count': 'number_of_train_stops'}) # service_count	
international_aero_availability = a['international_aero_availability'].rename(columns={'to_service': 'international_aero_availability'}) # to_service
number_of_international_aero = a['number_of_international_aero'].rename(columns={'service_count': 'number_of_international_aero'}) # service_count
local_aero_availability = a['local_aero_availability'].rename(columns={'to_service': 'local_aero_availability'}) # to_service
number_of_local_aero = a['number_of_local_aero'].rename(columns={'service_count': 'number_of_local_aero'}) # service_count
number_of_bus_stops = a['number_of_bus_stops'].rename(columns={'service_count': 'number_of_bus_stops'}) # service_count
train_paths_length = a['train_paths_length'].rename(columns={'total_length_km': 'train_paths_length'}) # total_length_km
number_of_bus_routes = a['number_of_bus_routes'].rename(columns={'number_of_routes': 'number_of_bus_routes'}) # number_of_routes
number_of_ports = a['number_of_ports'].rename(columns={'service_count': 'number_of_ports'}) 

In [229]:
train_paths_length

,id,index,name,train_paths_length,geometry
0,0,0,Тульская_область,2371.024053,"POLYGON ((296002.715 5971313.180, 296090.881 5..."


In [30]:
gdfs = [
    aggregated_density, road_length_gdf, to_fed_roads, to_region_admin_center,  azs_availability,
    number_of_fuel_stations, train_stops_availability, number_of_train_stops,
    international_aero_availability, number_of_international_aero, local_aero_availability,
    number_of_local_aero, number_of_bus_stops, train_paths_length, number_of_bus_routes, number_of_ports
]

target_crs = 'EPSG:4326'
for i, gdf in enumerate(gdfs):
    gdfs[i] = gdf.to_crs(target_crs)
    print(f"CRS for gdf {i}: {gdfs[i].crs}")

# Проверка всех преобразованных данных
for gdf in gdfs:
    print(gdf.head())

CRS for gdf 0: EPSG:4326
CRS for gdf 1: EPSG:4326
CRS for gdf 2: EPSG:4326
CRS for gdf 3: EPSG:4326
CRS for gdf 4: EPSG:4326
CRS for gdf 5: EPSG:4326
CRS for gdf 6: EPSG:4326
CRS for gdf 7: EPSG:4326
CRS for gdf 8: EPSG:4326
CRS for gdf 9: EPSG:4326
CRS for gdf 10: EPSG:4326
CRS for gdf 11: EPSG:4326
CRS for gdf 12: EPSG:4326
CRS for gdf 13: EPSG:4326
CRS for gdf 14: EPSG:4326
CRS for gdf 15: EPSG:4326
  id                name  road_density  \
0  0  Краснодарский_край      1018.414   

                                            geometry  
0  POLYGON ((36.53049 45.19920, 36.53270 45.18379...  
  id  index                name  reg1_length   reg2_length   reg3_length  \
0  0      0  Краснодарский_край  3485.151304  19206.410862  69651.115089   

                                            geometry  
0  POLYGON ((36.53049 45.19920, 36.53270 45.18379...  
  id  index                name  to_service  \
0  0      0  Краснодарский_край   19.848271   

                                         

In [276]:
gdfs = [
    aggregated_density, road_length_gdf, to_fed_roads, to_region_admin_center, connectivity,  azs_availability,
    number_of_fuel_stations, train_stops_availability, number_of_train_stops, international_aero_availability, number_of_international_aero, 
    local_aero_availability, number_of_local_aero, number_of_bus_stops, train_paths_length, number_of_bus_routes, number_of_ports
]

# Приведение всех геодатафреймов к одной CRS
target_crs = 'EPSG:4326'
for i, gdf in enumerate(gdfs):
    gdfs[i] = gdf.to_crs(target_crs)

# Создаем уникальный идентификатор для каждой строки в каждом GeoDataFrame
for i, gdf in enumerate(gdfs):
    gdf['unique_id'] = gdf.index

merged_gdf = gdfs[0]

# Объединяем все GeoDataFrame по колонке 'unique_id'
for gdf in gdfs[1:]:
    merged_gdf = merged_gdf.merge(gdf, on='unique_id', how='left', suffixes=('', f'_{gdf.name}'))

# Оставляем только нужные колонки, включая 'unique_id'
columns_to_keep = [
    'unique_id', 'geometry', 'name', 'road_density', 'reg1_length', 'reg2_length', 'reg3_length',
    'to_region_admin_center', 'connectivity', 'azs_availability', 'number_of_fuel_stations',
    'train_stops_availability', 'number_of_train_stops', 'international_aero_availability','number_of_international_aero',
        'local_aero_availability', 'number_of_local_aero',
    'number_of_bus_stops', 'train_paths_length', 'number_of_bus_routes', 'number_of_ports'
]

# Фильтруем финальный GeoDataFrame по выбранным колонкам
final_gdf = merged_gdf[columns_to_keep]

# Проверяем результат
final_gdf

,unique_id,geometry,name,road_density,reg1_length,reg2_length,reg3_length,to_region_admin_center,connectivity,azs_availability,...,train_stops_availability,number_of_train_stops,international_aero_availability,number_of_international_aero,local_aero_availability,number_of_local_aero,number_of_bus_stops,train_paths_length,number_of_bus_routes,number_of_ports
0,0,"MULTIPOLYGON (((70.35363 56.53420, 70.41604 56...",Ощепковское,546.034,0.000000,170.755587,91.063841,387.237642,3.745559,29.838300,...,73.612483,0.0,274.810967,0.0,76.295350,0.0,0.0,0.000000,0.0,0.0
1,1,"MULTIPOLYGON (((70.72792 56.52697, 70.72904 56...",Назаровское,193.166,0.000000,46.842710,18.292593,400.560750,3.955885,41.238317,...,86.935600,0.0,288.134083,0.0,89.618467,0.0,0.0,0.000000,0.0,0.0
2,2,"MULTIPOLYGON (((70.32875 56.18267, 70.32965 56...",Шевыринское,613.500,2.428493,101.437129,56.792285,376.589727,3.615178,12.700033,...,47.657733,0.0,264.968867,0.0,66.453250,0.0,0.0,0.000000,0.0,0.0
3,3,"MULTIPOLYGON (((69.89056 56.18638, 69.89624 56...",Тушнолобовское,553.875,42.695035,78.726380,34.897969,333.629402,3.081157,8.706917,...,31.729133,0.0,232.927617,0.0,34.412000,0.0,5.0,0.000000,0.0,0.0
4,4,"MULTIPOLYGON (((70.67055 56.08794, 70.67063 56...",Партизанское,334.199,0.000000,45.795596,34.092604,385.335220,3.760936,21.445517,...,65.729050,0.0,273.714350,0.0,75.198733,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,275,"MULTIPOLYGON (((68.14574 58.18468, 68.14770 58...",Тобольск,4026.144,32.305684,161.897215,772.077603,237.540921,3.449421,5.616208,...,14.179117,3.0,171.975425,0.0,14.833758,0.0,244.0,274.944423,0.0,5.0
276,276,"MULTIPOLYGON (((65.26049 57.28383, 65.26101 57...",Тюмень,5304.032,72.506249,450.740467,3174.702724,0.000000,2.949625,1.974883,...,1.621150,9.0,10.499117,1.0,6.137933,1.0,882.0,460.180282,16.0,0.0
277,277,"MULTIPOLYGON (((66.21807 56.64639, 66.22329 56...",Ялуторовск,7477.500,2.136434,47.818822,313.339983,80.101799,2.484423,2.884067,...,0.908083,3.0,64.204217,0.0,57.347750,0.0,121.0,29.419990,23.0,0.0
278,278,"MULTIPOLYGON (((66.08974 56.48680, 66.10479 56...",Заводоуковский,576.531,104.838981,789.531937,831.490666,117.439789,2.581777,11.479317,...,14.818900,8.0,92.168633,0.0,77.611750,0.0,175.0,189.627190,10.0,0.0


In [ ]:
final_gdf.explore('local_aero_availability')

In [279]:
final_gdf = final_gdf.drop(columns=['unique_id'])

In [280]:
final_gdf

,geometry,name,road_density,reg1_length,reg2_length,reg3_length,to_region_admin_center,connectivity,azs_availability,number_of_fuel_stations,train_stops_availability,number_of_train_stops,international_aero_availability,number_of_international_aero,local_aero_availability,number_of_local_aero,number_of_bus_stops,train_paths_length,number_of_bus_routes,number_of_ports
0,"MULTIPOLYGON (((70.35363 56.53420, 70.41604 56...",Ощепковское,546.034,0.000000,170.755587,91.063841,387.237642,3.745559,29.838300,0.0,73.612483,0.0,274.810967,0.0,76.295350,0.0,0.0,0.000000,0.0,0.0
1,"MULTIPOLYGON (((70.72792 56.52697, 70.72904 56...",Назаровское,193.166,0.000000,46.842710,18.292593,400.560750,3.955885,41.238317,0.0,86.935600,0.0,288.134083,0.0,89.618467,0.0,0.0,0.000000,0.0,0.0
2,"MULTIPOLYGON (((70.32875 56.18267, 70.32965 56...",Шевыринское,613.500,2.428493,101.437129,56.792285,376.589727,3.615178,12.700033,1.0,47.657733,0.0,264.968867,0.0,66.453250,0.0,0.0,0.000000,0.0,0.0
3,"MULTIPOLYGON (((69.89056 56.18638, 69.89624 56...",Тушнолобовское,553.875,42.695035,78.726380,34.897969,333.629402,3.081157,8.706917,1.0,31.729133,0.0,232.927617,0.0,34.412000,0.0,5.0,0.000000,0.0,0.0
4,"MULTIPOLYGON (((70.67055 56.08794, 70.67063 56...",Партизанское,334.199,0.000000,45.795596,34.092604,385.335220,3.760936,21.445517,0.0,65.729050,0.0,273.714350,0.0,75.198733,0.0,0.0,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
275,"MULTIPOLYGON (((68.14574 58.18468, 68.14770 58...",Тобольск,4026.144,32.305684,161.897215,772.077603,237.540921,3.449421,5.616208,18.0,14.179117,3.0,171.975425,0.0,14.833758,0.0,244.0,274.944423,0.0,5.0
276,"MULTIPOLYGON (((65.26049 57.28383, 65.26101 57...",Тюмень,5304.032,72.506249,450.740467,3174.702724,0.000000,2.949625,1.974883,77.0,1.621150,9.0,10.499117,1.0,6.137933,1.0,882.0,460.180282,16.0,0.0
277,"MULTIPOLYGON (((66.21807 56.64639, 66.22329 56...",Ялуторовск,7477.500,2.136434,47.818822,313.339983,80.101799,2.484423,2.884067,5.0,0.908083,3.0,64.204217,0.0,57.347750,0.0,121.0,29.419990,23.0,0.0
278,"MULTIPOLYGON (((66.08974 56.48680, 66.10479 56...",Заводоуковский,576.531,104.838981,789.531937,831.490666,117.439789,2.581777,11.479317,8.0,14.818900,8.0,92.168633,0.0,77.611750,0.0,175.0,189.627190,10.0,0.0


In [261]:
final_gdf['name'] = final_gdf['name'].replace('Tyumen Oblast', 'Тюменская область')

In [281]:
final_gdf.to_parquet('Тюменская_область_settlements.parquet')

In [2]:
from shapely.geometry import GeometryCollection

In [6]:
points18 = gpd.read_file('/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/data/Московская_область/Московская_область_region_points.geojson')
points18 = points18[points18['is_admin_centre_district']==True]


In [2]:
frame = nx.read_graphml("/Users/sashamorozov/Documents/НЦКР платформа ЦУ/Показатели по 9 регионам/data/graphs/Ленинградская_область/Ленинградская_область_uds.graphml")
frame = graphbuilder.convert_list_attr_from_str(frame)
frame = indicators.prepare_graph(frame)
# frame = graphbuilder.assign_city_names_to_nodes(points18.reset_index().to_crs(frame.graph['crs']), 
#                                                 momepy.nx_to_gdf(frame)[0], frame, name_col='name', max_distance=1200)

n,e = momepy.nx_to_gdf(frame)

In [3]:
nodes_layer = gpd.read_file('/Users/sashamorozov/Downloads/frame_nodes_spb_lo.geojson')
edges_layer = gpd.read_file('/Users/sashamorozov/Downloads/frame_edges_spb_lo.geojson')

nodes_layer = nodes_layer.to_crs('EPSG:4326')
edges_layer = edges_layer.to_crs('EPSG:4326')

In [4]:
n = n.to_crs('EPSG:4326')
e = e.to_crs('EPSG:4326')

In [5]:
n

,reg_1,reg_2,x,y,nodeID,exit,exit_country,geometry
0,False,False,316822.011829,6.514139e+06,0,NaN,NaN,POINT (29.83578 58.72844)
1,False,False,316864.213100,6.514292e+06,1,NaN,NaN,POINT (29.83639 58.72983)
2,False,False,317015.043435,6.514061e+06,2,NaN,NaN,POINT (29.83918 58.72782)
3,False,False,316716.593408,6.514168e+06,3,NaN,NaN,POINT (29.83394 58.72866)
4,False,False,316771.697024,6.513968e+06,4,NaN,NaN,POINT (29.83505 58.72689)
...,...,...,...,...,...,...,...,...
126388,False,False,222720.907545,6.728057e+06,126388,NaN,NaN,POINT (27.93469 60.59223)
126389,False,False,222481.292591,6.728106e+06,126389,NaN,NaN,POINT (27.93026 60.59251)
126390,False,False,222429.409497,6.727811e+06,126390,NaN,NaN,POINT (27.92973 60.58983)
126391,False,False,222453.683954,6.727778e+06,126391,NaN,NaN,POINT (27.93022 60.58955)


In [39]:
n.city_name.unique()

array([nan, 'Луга', 'Гатчина', 'Волосово', 'Сосновый бор', 'Всеволожск',
       'Приозерск', 'Кировск', 'Кириши', 'Тосно', 'Волхов',
       'Лодейное Поле', 'Тихвин', 'Бокситогорск', 'Подпорожье',
       'Кингисепп', 'Выборг', 'Сланцы'], dtype=object)

In [6]:
combined_gdf = gpd.GeoDataFrame(pd.concat([n, e], ignore_index=True))

In [ ]:
combined_gdf.explore()

In [8]:
def tolist(var):
    if isinstance(var,list):
        return str(var)
    return var
combined_gdf= combined_gdf.applymap(tolist)

/var/folders/zx/byb3bcds50s1lwbdr2mwq18r0000gn/T/ipykernel_1561/2016932770.py:5: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  combined_gdf= combined_gdf.applymap(tolist)


In [9]:
combined_gdf.to_file('uds_LO.gpkg', driver='GPKG')

In [2]:
import osmnx as ox
import geopandas as gpd
import pandas as pd

# Определяем имя области
reg = 'Московская область, Russia'

# Загружаем границы области
gdf = ox.geocode_to_gdf(reg)

# Устанавливаем границы области как фильтр
poly = gdf.unary_union

# Загружаем точки населенных пунктов
tags = {'place': True}
pois = ox.geometries.geometries_from_polygon(poly, tags)

# Фильтруем только узлы (точки)
pois_nodes = pois[pois.geometry.type == 'Point']

# Создаем две новые колонки, заполняя их значением False по умолчанию
pois_nodes['is_admin_centre_district'] = False

/var/folders/zx/byb3bcds50s1lwbdr2mwq18r0000gn/T/ipykernel_4009/1735054474.py:16: FutureWarning: The `geometries` module and `geometries_from_X` functions have been renamed the `features` module and `features_from_X` functions. Use these instead. The `geometries` module and function names are deprecated and will be removed in the v2.0.0 release. See the OSMnx v2 migration guide: https://github.com/gboeing/osmnx/issues/1123
  pois = ox.geometries.geometries_from_polygon(poly, tags)
/Users/sashamorozov/Documents/Code/NCCR/transport_frames/.venv/lib/python3.10/site-packages/osmnx/_overpass.py:254: UserWarning: This area is 24 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


In [21]:
pois_nodes.explore()

In [6]:
pois_nodes

addr:country addr:postcode         addr:region  \
element_type osmid                                                       
node         306545784            RU        142970  Московская область   
             336306248            RU           NaN  Московская область   
             359050294            RU           NaN  Московская область   
             359050306            RU           NaN  Московская область   
             360616724            RU        142960  Московская область   
...                              ...           ...                 ...   
             2618381051           RU        143824  Московская область   
             2633505515           RU        143821  Московская область   
             2638313752           RU        143821  Московская область   
             2638313755           RU        143801  Московская область   
             2641934446           RU        143813  Московская область   

                                                               description  \
element_type osmid                                                           
node         306545784   посёлок городского типа областного подчинения ...   
             336306248                                                 NaN   
             359050294                                                 NaN   
             359050306                                                 NaN   
             360616724                                                 NaN   
...                                                                    ...   
             2618381051                                                NaN   
             2633505515                                                NaN   
             2638313752                                                NaN   
             2638313755                                                NaN   
             2641934446                                                NaN   

                                     name           name:ar  \
element_type osmid                                            
node         306545784   Серебряные Пруды  سيريبرينيي برودي   
             336306248           Подхожее               NaN   
             359050294           Шеметово               NaN   
             359050306      Александровка               NaN   
             360616724            Узуново               NaN   
...                                   ...               ...   
             2618381051           Мазлово               NaN   
             2633505515          Себудово               NaN   
             2638313752        Введенское               NaN   
             2638313755      Новолотошино               NaN   
             2641934446          Татьянки               NaN   

                                    name:de            name:en  \
element_type osmid                                               
node         306545784   Serebrjanyje Prudy  Serebryanye Prudy   
             336306248                  NaN        Podkhozheye   
             359050294                  NaN                NaN   
             359050306                  NaN                NaN   
             360616724                  NaN            Uzunovo   
...                                     ...                ...   
             2618381051                 NaN                NaN   
             2633505515                 NaN                NaN   
             2638313752                 NaN                NaN   
             2638313755                 NaN                NaN   
             2641934446                 NaN                NaN   

                                   name:es            name:it  ...  \
element_type osmid                                             ...   
node         306545784   Serébrianye Prudý  Serebrjanye Prudy  ...   
             336306248                 NaN                NaN  ...   
             359050294                 NaN                NaN  ...   
             3590

In [23]:
# Список городов России
cities = [
    "Балашиха",
    "Ногинск",
    "Бронницы",
    "Власиха",
    "Волоколамск",
    "Воскресенск",
    "Восход",
    "Дзержинский",
    "Дмитров",
    "Долгопрудный",
    "Домодедово",
    "Дубна",
    "Егорьевск",
    "Жуковский",
    "Зарайск",
    "Истра",
    "Кашира",
    "Клин",
    "Коломна",
    "Королёв",
    "Котельники",
    "Красногорск",
    "Краснознаменск",
    "Видное",
    "Лобня",
    "Лосино-Петровский",
    "Лотошино",
    "Луховицы",
    "Лыткарино",
    "Люберцы",
    "Можайск",
    "Звездный городок",
    "Мытищи",
    "Наро-Фоминск",
    "Одинцово",
    "Орехово-Зуево",
    "Электрогорск",
    "Подольск",
    "Протвино",
    "Пушкино",
    "Серпухов",
    "Раменское",
    "Реутов",
    "Руза",
    "Сергиев Посад",
    "Серебряные Пруды",
    "Пущино",
    "Солнечногорск",
    "Ступино",
    "Талдом",
    "Фрязино",
    "Химки",
    "Черноголовка",
    "Чехов",
    "Шатура",
    "Шаховская",
    "Щелково",
    "Павловский Посад",
    "Электросталь",
    "Молодёжный"
]


# Создаем список городов
city_list = cities

# Выводим список городов
print(city_list)


['Балашиха', 'Ногинск', 'Бронницы', 'Власиха', 'Волоколамск', 'Воскресенск', 'Восход', 'Дзержинский', 'Дмитровский', 'Долгопрудный', 'Домодедово', 'Дубна', 'Егорьевск', 'Жуковский', 'Зарайск', 'Истра', 'Кашира', 'Клин', 'Коломна', 'Королёв', 'Котельники', 'Красногорск', 'Краснознаменск', 'Ленинский', 'Лобня', 'Лосино-Петровский', 'Лотошино', 'Луховицы', 'Лыткарино', 'Люберцы', 'Можайск', 'Звездный городок', 'Мытищи', 'Наро-Фоминск', 'Одинцово', 'Орехово-Зуево', 'Электрогорск', 'Подольск', 'Протвино', 'Пушкино', 'Серпухов', 'Раменский', 'Реутов', 'Рузский', 'Сергиево-Посадский', 'Серебряные Пруды', 'Пущино', 'Солнечногорский', 'Ступино', 'Талдомский', 'Фрязино', 'Химки', 'Черноголовка', 'Чехов', 'Шатура', 'Шаховская', 'Щелково', 'Павловвский Посад', 'Электросталь', 'Молодёжный']


In [24]:
# список административных центров
pois_nodes['is_admin_centre_district'] = pois_nodes['name'].isin(city_list)


In [25]:
pois_nodes = pois_nodes[["name","geometry","is_admin_centre_district"]]

In [27]:
pois_nodes.to_file('moscow_np.geojson', driver='GeoJSON')

In [22]:
len(city_list)

60